# Retriever
- 비정형 질의(query)를 입력 받아 Vector store에서 관련된 내용을 **검색하는 기능**을 제공하는 인터페이스
- 다양한 데이터 소스에서 정보를 검색하여 대규모 언어 모델(LLM) 기반 애플리케이션의 **정확성을** 향상시키는 데 핵심적인 역할을 한다.

![RAG](figures/rag2.png)

## 주요 특징
- **다양한 데이터 소스 지원**
	- Retriever는 벡터 스토어, 그래프 데이터베이스, 관계형 데이터베이스 등 여러 종류의 검색 시스템과 상호작용할 수 있는 통일된 인터페이스를 제공한다다.
- **간단한 인터페이스**: Retriever는 문자열 형태의 쿼리를 입력받아 관련 문서의 리스트를 반환한다. 이러한 단순한 구조 덕분에 다양한 검색 시스템과 쉽게 통합할 수 있다. 


## 다양한 Retriever 방식

**Retriever**란, 사용자의 질문(쿼리)에 가장 관련성 높은 정보를 찾아주는 구성 요소이다. 주로 검색 기반 질문응답 시스템(RAG, Retrieval-Augmented Generation)에서 사용된다. 다음은 자주 사용되는 다양한 Retriever의 유형과 그 특징이다.

1. **벡터 스토어(Vector Store) Retriever**
   - VectorStore로 부터 유사도를 기반으로 검색하는 가장 기본 Retriever
   - 텍스트 조각(청크)마다 **임베딩(embedding)을** 생성하여 벡터 공간에 저장하고, 쿼리 임베딩과의 **코사인 유사도(cosine similarity)** 등을 기반으로 유사한 텍스트를 검색한다.
   - 검색 속도가 빠르고 구현이 간단하여, 기본적인 검색 시스템을 구축할 때 적합하다.
2. **[ParentDocumentRetriever](https://python.langchain.com/docs/how_to/parent_document_retriever/)**
   - **하나의 문서를 여러 청크**로 나눈 뒤 각각을 인덱싱하고, 쿼리와 가장 유사한 청크를 찾은 다음 해당 청크가 속한 **전체 원본 문서**를 반환한다.
   - 정확도가 올라가겠죠 ?
   - 작은 정보 조각들이 모여 하나의 문서를 구성할 때 유용하며, 문맥을 넓게 유지할 수 있다.
3. **[MultiVectorRetriever](https://python.langchain.com/docs/how_to/multi_vector/)**
   - 각 문서에 대해 요약을 하거나, 가상의 질문을 생성하거나, 사람이 중요한 내용을 직접 추가하여 **문서당 여러 개의 임베딩 벡터**를 생성한다.
   - 텍스트 전체보다 더 핵심적인 정보가 검색에 반영되도록 하고자 할 때 효과적이다.
   - 특히, 문서가 길거나, 중요한 내용이 문서의 특정 부분에 집중되어 있는 경우에 유리하다.
4. **[SelfQueryRetriever](https://python.langchain.com/docs/how_to/self_query/)**
   - 대규모 언어 모델(LLM, Large Language Model)을 활용하여 사용자의 질문을 적절한 검색어와 **메타데이터(metadata)** 필터로 자동 변환한다.
   - 예를 들어, 문서의 작성자, 날짜, 태그와 같은 메타데이터를 기준으로 검색할 수 있다.
   - 문서 자체의 내용뿐만 아니라, 문서에 부가된 속성 정보에 대해 질문할 때 유용하다.
5. **[ContextualCompressionRetriever](https://python.langchain.com/docs/how_to/contextual_compression/)**
   - 기존 Retriever와 조합되어 사용된다.
   - 먼저 일반적인 검색을 수행한 후, 검색된 문서들에서 쿼리와 관련 없는 불필요한 정보를 제거하고 핵심 내용만을 추출하여 반환한다.
   - 정보를 요약하거나 압축하여 LLM에 전달할 문서 길이를 줄일 때 유용하다.
6. **[MultiQueryRetriever](https://python.langchain.com/docs/how_to/MultiQueryRetriever/)**
   - LLM을 이용해 하나의 쿼리로부터 여러 가지 변형된 쿼리를 생성하고, 각 쿼리에 대해 검색을 수행한 뒤 결과를 합치는 방식.
   - 다양한 표현에 강해 검색 범위를 넓히고 성능을 높인다.
7. **[EnsembleRetriever](https://python.langchain.com/docs/how_to/ensemble_retriever/)**
   - 여러 개의 Retriever(예: BM25, 벡터 기반 등)를 결합해 가중치를 기반으로 결과를 조합(re-ranking)한다.
   - 서로 다른 장점을 가진 Retriever를 하나로 묶어 성능을 강화한다.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# Load -> chunking -> embedding -> store
text_path = 'data/olympic.txt'
collection_name = "olympic_info"
persist_directory = "vector_store/chroma/olympic_info"

# 1. load + split
loader = TextLoader(text_path, encoding='utf-8')
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)

docs = loader.load_and_split(splitter)
print(len(docs))

In [ ]:
# VectorStore와 연결
embedding_model = OpenAIEmbeddings(model = "text-embedding-3-large")
# vector_store = Chroma.from_documents(docs = docs	# 연결하면서 문서 추가)
vector_store = Chroma(	# 연결만.
    embedding_function=embedding_model,
    collection_name=collection_name,
    persist_directory=persist_directory
)

In [ ]:
# Vector DB에 문서 추가.
add_ids = vector_store.add_documents(docs)

In [ ]:
# add_ids
vector_store._collection.count()

In [ ]:
# VectorStore(Chroma DB와 연결)로부터 검색(Retrieve)하는 Retriever 생성
retriever = vector_store.as_retriever()
result_docs = retriever.invoke("올림픽과 관련된 논란들은 무엇이 있나요 ?")	# 검색할 query를 str로 전달, default로 4개 출력.

In [ ]:
result_docs

In [ ]:
# 검색 설정을 넣어서 retriever 생성
retriever2 = vector_store.as_retriever(
    search_type = "similarity",		# "similarity_score_threshold", "mmr", default : "similarity"
    search_kwargs = {"k" : 10,		# 검색 method(similarity_search_xxxx())의 parameter를 dictionary로 전달.
                    #  "filter" : {"source" : "abc"}
	}
)
result_docs2 = retriever2.invoke("올림픽과 관련된 논란들은 무엇이 있나요 ?")
result_docs2

In [ ]:
# 검색 설정을 넣어서 retriever 생성
retriever3 = vector_store.as_retriever(
    search_type = "similarity_score_threshold",		# "similarity_score_threshold", "mmr", default : "similarity"
    search_kwargs = {"k" : 10, "score_threshold" : 0.3}		# 유사도 점수가 지정한 값 이상인 것만 조회
)
result_docs3 = retriever3.invoke("올림픽과 관련된 논란들은 무엇이 있나요 ?")
result_docs3

In [ ]:
# 검색 설정을 넣어서 retriever 생성
retriever4 = vector_store.as_retriever(
    search_type = "mmr",		# "similarity_score_threshold", "mmr", default : "similarity"
    search_kwargs = {"k" : 5, "fetch_k" : 20, "lambda_mult" : 0.5}		# mmr 파라미터
)
result_docs4 = retriever4.invoke("올림픽과 관련된 논란들은 무엇이 있나요 ?")
result_docs4

In [ ]:
# retriever | 질문과 검색결과를 prompt로 생성 | LLM | output parser => chain
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

In [ ]:
# RAG용 prompt template : "답변을 context 기반으로 해야한다"라는 system message가 들어가야함.
template = """
# Instruction:
당신은 정확한 정보 제공을 우선시하는 인공지능 어시스턴트입니다.
주어진 Context에 포함된 정보만 사용해서 질문에 답변하세요.
Context에 질문에 대한 명확한 정보가 있는 경우 그 내용을 바탕으로 답변하세요.
Context에 질문에 대한 명확한 정보없을 경우 "정보가 부족해서 답을 알 수 없습니다." 라고 대답합니다.
절대 Context에 없는 내용을 추측하거나 일반 상식을 이용해 답을 만들어서 대답하지 않습니다.

Context:
{context}

질문:
{query}
"""
prompt_template = PromptTemplate(template=template)

In [ ]:
from langchain_core.documents import Document
def format_docs(docs:list[Document]) -> str:
    """
    Retriever가 검색한 문서들에서 page_content(문서 내용)만 추출해서 반환
    추출된 문서들의 내용을 "\n\n"으로 연결한다.
    Args:
		docs(list[Document]) - 검색한 문서 리스트
	Returns:
		str - 문서1내용+\n\n+문서2내용+\n\n+...
    """
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# format_docs(result_docs)

In [ ]:
# 처음엔 아래 cell처럼 chain 구성하기 힘듬. 그래서 이런식으로 func을 활용해도 좋음.
model = ChatOpenAI(model_name = "gpt-4.1-mini")

from langchain_core.runnables import chain
@chain
def final_chain(query : str):
    docs = retriever.invoke(query)
    docs = format_docs(docs)
    prompt = prompt_template.invoke({"context" : docs, "query" : query})
    result = model.invoke(prompt)
    return StrOutputParser().invoke(result)

final_chain.invoke("올림픽에 대해 설명해줘")

In [ ]:
# chain 구성 : {retriever, 질문} -> prompt_template -> llm -> output parser
model = ChatOpenAI(model_name = "gpt-4.1-mini")

# func의 parameter가 하나면 이런식으로 묶어도 됨.
# {"context" : Str} 형태로 나옴.
chain = ({"context" : retriever | format_docs, "query" : RunnablePassthrough()} 
         | prompt_template
         | model
         | StrOutputParser()
         )

In [ ]:
# 질의
query = "올림픽과 관련된 논란에 대해 알려줘."
result = chain.invoke(query)
print(result)

In [ ]:
chain.invoke("롤에 대해 알려줘")

# TODO 다음을 작성한다.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


collection_name = "olympic_docs"
persist_directory = "vector_store/chroma/olympic"

# Text Loading


# Split


# Vector Store 생성


# Retriever 생성 - "mmr" 방식



In [ ]:
# Prompt Template 생성



# Chain 구성




In [ ]:
# Chain을 이용해 질의


